# DDI KG

## Check Python Version

In [1]:
!python --version

Python 3.11.9


## Setup Config

In [2]:
import pathlib

PROJECT_PATH = "../.."
AUDIT_PATH = "../../datasets/audit"

DATASET_DIR = pathlib.Path(DATASET_PATH)
print(f"Dataset Directory: {DATASET_DIR}")
AUDIT_DIR = pathlib.Path(AUDIT_PATH)
print(f"Audit Directory: {AUDIT_DIR}")

print("="*10)

RX_INGREDIENT = "rx_norm_ingredients.csv"
RX_PRECISE_INGREDIENT = "rx_norm_master_precise_ingredients.csv"
RX_SYNONYM_INGREDIENT = "rx_norm_ingredient_synonyms.csv"
RX_PRECISE_SYNONYM_INGREDIENT = "rx_norm_precise_ingredients_synonyms.csv"

RX_INGREDIENT_PATH = DATASET_DIR.joinpath(RX_INGREDIENT)
RX_PRECISE_INGREDIENT_PATH = DATASET_DIR.joinpath(RX_PRECISE_INGREDIENT)
RX_SYNONYM_INGREDIENT_PATH = DATASET_DIR.joinpath(RX_SYNONYM_INGREDIENT)
RX_SYNONYM_PIN_PATH = DATASET_DIR.joinpath(RX_PRECISE_SYNONYM_INGREDIENT)

print(f"RX Ingredients dataset Location: {RX_INGREDIENT_PATH}")
print(f"RX Precise Ingredients dataset Location: {RX_PRECISE_INGREDIENT_PATH}")
print(f"RX Synonym Ingredients dataset Location: {RX_SYNONYM_INGREDIENT_PATH}")
print(f"RX Synonym Precise Ingredient dataset Location: {RX_SYNONYM_PIN_PATH}")

print("="*10)

DDI = "db_drug_interactions.csv"
DDI_PATH = DATASET_DIR.joinpath(DDI)
print(f"DDI Records Location: {DDI_PATH}")

BPOM_BRAND = "bpom_drug_cleaned.csv"
BPOM_BRAND_PATH = DATASET_DIR.joinpath(BPOM_BRAND)
print(f"BPOM Product Records Location: {BPOM_BRAND_PATH}")

print("="*10)

SKIPPED_DDI_RECORD_LOG = "skipped_ddi_records.csv"
SKIPPED_DDI_RECORD_LOG_PATH = AUDIT_DIR.joinpath(SKIPPED_DDI_RECORD_LOG)
print(f"Skipped DDI Records Log Location: {SKIPPED_DDI_RECORD_LOG_PATH}")

UNRESOLVED_BRAND_ING_LOG = "unresolved_brand_ingredients.csv"
UNRESOLVED_BRAND_ING_LOG_PATH = AUDIT_DIR.joinpath(UNRESOLVED_BRAND_ING_LOG)
print(f"Unresolved Brand Ingredients Log Location: {UNRESOLVED_BRAND_ING_LOG_PATH}")

DUPLICATE_ALIAS_SY = "duplicate_rx_synonyms.csv"
DUPLICATE_ALIAS_SY_PATH = AUDIT_DIR.joinpath(DUPLICATE_ALIAS_SY)
print(f"Duplicate Synonyms Rx Entity Log Location: {DUPLICATE_ALIAS_SY_PATH}")

Dataset Directory: ..\..\datasets
Audit Directory: ..\..\datasets\audit
RX Ingredients dataset Location: ..\..\datasets\rx_norm_ingredients.csv
RX Precise Ingredients dataset Location: ..\..\datasets\rx_norm_master_precise_ingredients.csv
RX Synonym Ingredients dataset Location: ..\..\datasets\rx_norm_ingredient_synonyms.csv
RX Synonym Precise Ingredient dataset Location: ..\..\datasets\rx_norm_precise_ingredients_synonyms.csv
DDI Records Location: ..\..\datasets\db_drug_interactions.csv
BPOM Product Records Location: ..\..\datasets\bpom_drug_cleaned.csv
Skipped DDI Records Log Location: ..\..\datasets\audit\skipped_ddi_records.csv
Unresolved Brand Ingredients Log Location: ..\..\datasets\audit\unresolved_brand_ingredients.csv
Duplicate Synonyms Rx Entity Log Location: ..\..\datasets\audit\duplicate_rx_synonyms.csv


## RxNorm Analysis

In [3]:
from neo4j import GraphDatabase

URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "password123"

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

query = """
MATCH (n)
WHERE n:Ingredient OR n:PreciseIngredient OR n:Synonym
RETURN
    count(CASE
        WHEN n:Ingredient AND n.rxcui IS NOT NULL
        THEN 1
    END) AS ingredients,

    count(CASE
        WHEN n:PreciseIngredient
        THEN 1
    END) AS precise_ingredients,

    count(CASE
        WHEN n:Synonym
        THEN 1
    END) AS synonyms
"""

with driver.session() as session:
    record = session.run(query).single()

node_in = record['ingredients']
node_pin = record['precise_ingredients']
node_sy = record['synonyms']

driver.close()

Received notification from DBMS server: <GqlStatusObject gql_status='01G11', status_description='warn: null value eliminated in set function', position=None, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (n)\nWHERE n:Ingredient OR n:PreciseIngredient OR n:Synonym\nRETURN\n    count(CASE\n        WHEN n:Ingredient AND n.rxcui IS NOT NULL\n        THEN 1\n    END) AS ingredients,\n\n    count(CASE\n        WHEN n:PreciseIngredient\n        THEN 1\n    END) AS precise_ingredients,\n\n    count(CASE\n        WHEN n:Synonym\n        THEN 1\n    END) AS synonyms\n'


In [4]:
import pandas as pd
from typing import Dict, List, Set, Tuple
import ast

in_synonyms_count = 0
pin_synonyms_count = 0

rx_sy_in_df = pd.read_csv(RX_SYNONYM_INGREDIENT_PATH)
rx_sy_pin_df = pd.read_csv(RX_SYNONYM_PIN_PATH)


class AliasesListCounter:

    def parse_aliases(self, raw_aliases) -> List[Dict[str, str]]:
        if not raw_aliases or str(raw_aliases).lower() == "nan":
            return []

        if isinstance(raw_aliases, list):
            return raw_aliases

        if isinstance(raw_aliases, str):
            try:
                parsed = ast.literal_eval(raw_aliases)
                if isinstance(parsed, list):
                    return parsed
            except (ValueError, SyntaxError):
                pass

        return []

    def count_aliases(self, df: pd.DataFrame) -> int:
        counter = 0

        for _, row in df.iterrows():
            aliases = self.parse_aliases(row.get("aliases"))
            counter += len(aliases)

        return counter

    def get_unique_aliases(
        self,
        df: pd.DataFrame,
    ) -> Set[Tuple[str, str]]:
        unique = set()

        for _, row in df.iterrows():
            aliases = self.parse_aliases(row.get("aliases"))

            for alias in aliases:
                if not isinstance(alias, dict):
                    continue

                rx_cui = str(alias.get("rx_cui", "")).strip()
                name = str(alias.get("name", "")).strip()
                sources = alias.get("source", [])

                for source in sources:
                    unique.add((name, source))

        return unique


counter = AliasesListCounter()

in_synonyms_count = counter.count_aliases(rx_sy_in_df)
pin_synonyms_count = counter.count_aliases(rx_sy_pin_df)

unique_in_synonyms = counter.get_unique_aliases(rx_sy_in_df)
unique_pin_synonyms = counter.get_unique_aliases(rx_sy_pin_df)

unique_synonyms = unique_in_synonyms | unique_pin_synonyms

In [5]:
import pandas as pd
from typing import List, Dict, Any
import ast

rx_in_df = pd.read_csv(RX_INGREDIENT_PATH)
rx_pre_in_df = pd.read_csv(RX_PRECISE_INGREDIENT_PATH)

print(20*"=")
print(f"Rx Ingredient total rows: {rx_in_df.shape[0]}")
print(f"Rx Ingredient node created: {node_in}")
created_in = (node_in/rx_in_df.shape[0]) * 100
print(f"Ingredient successfully ingested: {created_in:.2f}%")

print(20*"=")
print(f"Rx Precise Ingredient total rows: {rx_pre_in_df.shape[0]}")
print(f"Rx Precise Ingredient Node Count: {node_pin}")
created_pin = (node_pin/rx_pre_in_df.shape[0]) * 100
print(f"Precise Ingredient successfully ingested: {created_pin:.2f}%")

print(20*"=")
print(f"Rx Synonym Total Rows: {in_synonyms_count + pin_synonyms_count}")
print(f"Rx Unique Synonym Total: { len(unique_synonyms) }")
print(f"Rx Synonym Node Count: {node_sy}")
created_sy = (node_sy/len(unique_synonyms)) * 100
print(f"Synonym successfully ingested: {created_sy:.2f}%")


Rx Ingredient total rows: 14663
Rx Ingredient node created: 14663
Ingredient successfully ingested: 100.00%
Rx Precise Ingredient total rows: 3659
Rx Precise Ingredient Node Count: 3659
Precise Ingredient successfully ingested: 100.00%
Rx Synonym Total Rows: 20708
Rx Unique Synonym Total: 20977
Rx Synonym Node Count: 20977
Synonym successfully ingested: 100.00%


## Check Duplicate Synonyms

### Duplicate Synonyms from Dataset

In [6]:
import ast
import pandas as pd


class SynonymDuplicateChecker:
    
    def _parse_aliases(self, value):
        if pd.isna(value) or value == "[]":
            return []
    
        if isinstance(value, list):
            return value
    
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else []
        except (ValueError, SyntaxError):
            return []
    
    
    def get_duplicate_aliases(
        self,
        df_ingredient: pd.DataFrame,
        df_precise: pd.DataFrame,
    ):
        frames = []
    
        for df, dataset in [
            (df_ingredient, "IN"),
            (df_precise, "PIN"),
        ]:
            tmp = df[["rxcui", "name", "aliases"]].copy()
    
            tmp["aliases"] = tmp["aliases"].apply(self._parse_aliases)
            tmp = tmp.explode("aliases")
    
            # Ambil nama alias dari JSON object
            tmp["alias"] = tmp["aliases"].apply(
                lambda x: x.get("name")
                if isinstance(x, dict)
                else None
            )
    
            tmp["dataset"] = dataset
    
            frames.append(
                tmp[
                    ["rxcui", "name", "alias", "dataset"]
                ]
            )
    
        all_aliases = pd.concat(
            frames,
            ignore_index=True
        )
    
        # Buang alias kosong
        all_aliases = all_aliases[
            all_aliases["alias"].notna()
            & all_aliases["alias"].astype(str).str.strip().ne("")
        ]
    
        # Cari alias yang muncul > 1 kali
        alias_counts = all_aliases["alias"].value_counts()
    
        duplicate_names = alias_counts[
            alias_counts > 1
        ].index
    
        duplicates = all_aliases[
            all_aliases["alias"].isin(duplicate_names)
        ]
    
        # Summary
        result = (
            duplicates
            .groupby("alias", as_index=False)
            .agg(
                rxcui_resolved=(
                    "rxcui",
                    lambda x: sorted(
                        set(x.astype(str))
                    )
                ),
                dataset=(
                    "dataset",
                    lambda x: sorted(set(x))
                ),
                occurrences=("alias", "size"),
            )
            .sort_values("alias")
            .reset_index(drop=True)
        )
    
        return result

In [7]:
import ast
import pandas as pd


df_ingredient = pd.read_csv(RX_SYNONYM_INGREDIENT_PATH)
df_precise = pd.read_csv(RX_SYNONYM_PIN_PATH)

checker = SynonymDuplicateChecker()

duplicates = checker.get_duplicate_aliases(df_ingredient, df_precise)
duplicates

,alias,rxcui_resolved,dataset,occurrences
0,corticotropin-releasing factor,"[1927940, 214461]","[IN, PIN]",2
1,sterculia,"[6111, 904554]",[IN],2
2,tapioca starch,"[2619252, 901346]",[IN],2


In [8]:
import pathlib

DATASET_PATH = "../../datasets/"
REPORT_PATH = "../../datasets/knowledge_graph"

DATASET_DIR = pathlib.Path(DATASET_PATH)
print(f"Dataset Directory: {DATASET_DIR}")

REPORT_DIR = pathlib.Path(REPORT_PATH)
print(f"Dataset Report Directory: {DATASET_DIR}")

print("="*10)

Dataset Directory: ..\..\datasets
Dataset Report Directory: ..\..\datasets


### Duplicate SY in Knowledge Graph

In [9]:
def print_relationships(df):
    for _, row in df.iterrows():
        print("=" * 70)

        # Source node
        print(format_node(row["s"]))

        # Relationship
        print("      │")
        print(f"      └── {row['r']}")
        print("                  │")
        print("                  ▼")

        # Target node
        print(format_node(row["n"]))

        print()


def format_node(node):
    """
    Convert Neo4j node representation into a readable format.
    """
    node = str(node)

    # Example:
    # (:Synonym {name: foo,source: RxNav_RXNORM,id: syn_xxx})
    # -> multiline readable representation

    if "{" not in node:
        return node

    label_part, properties_part = node.split("{", 1)

    label = label_part.strip()
    properties = properties_part.rstrip("}")

    # Split properties by comma
    props = []

    for item in properties.split(","):
        if ":" not in item:
            continue

        key, value = item.split(":", 1)

        props.append(
            f"  {key.strip():10}: {value.strip()}"
        )

    return "\n".join(
        [label] + props
    )

In [10]:
import pandas as pd

DUPLICATE_SY = [
    "corticotropin-releasing_factor",
    "sterculia",
    "tapioca_starch"
]
duplicate_synonyms = {}
for syn in DUPLICATE_SY:
    path = REPORT_DIR.joinpath(f"{syn}.csv")
    
    df = pd.read_csv(path)
    
    duplicate_synonyms[syn] = df

#### Corticotropin-Releasing Factor

In [11]:
 print_relationships(duplicate_synonyms["corticotropin-releasing_factor"])

(:Synonym
  name      : corticotropin-releasing factor
  source    : RxNav_RXNORM
  id        : syn_4175fcf11b82eeb3a1a1a0e500eca9f3dbb3fb2064628d414a44adee34eba634})
      │
      └── [:HAS_SYNONYM]
                  │
                  ▼
(:Ingredient
  name      : corticotropin-releasing hormone
  source    : RxNorm
  rxcui     : 214461})

(:Synonym
  name      : corticotropin-releasing factor
  source    : RxNav_DRUGBANK
  id        : syn_8d77d7f1a6a8a40b80b73ffc75fb1317685a1366f8eb2b1edde3810226bed59d})
      │
      └── [:HAS_SYNONYM]
                  │
                  ▼
(:PreciseIngredient
  name      : corticotropin-releasing factor
  source    : RxNorm
  rxcui     : 1927940})



![corticotropin-releasing_factor](../../datasets/knowledge_graph/images/corticotropin-releasing_factor.png)

#### Sterculia

In [12]:
 print_relationships(duplicate_synonyms["sterculia"])

(:Synonym
  name      : sterculia
  source    : RxNav_RXNORM
  id        : syn_46df392f4fd6aeaa11ace65434d6c1744e8a4aa9b3c96510993f84c41956ade9})
      │
      └── [:HAS_SYNONYM]
                  │
                  ▼
(:Ingredient
  name      : karaya gum
  source    : RxNorm
  rxcui     : 6111})

(:Synonym
  name      : sterculia
  source    : RxNav_DRUGBANK
  id        : syn_44ac76379e7eb90de7d775a4597b5537b3d8991bfd2784400c0747e442f6c74a})
      │
      └── [:HAS_SYNONYM]
                  │
                  ▼
(:Ingredient
  name      : karaya gum allergenic extract
  source    : RxNorm
  rxcui     : 904554})



![sterculia](../../datasets/knowledge_graph/images/sterculia.png)

#### Tapioca Starch

In [13]:
print_relationships(duplicate_synonyms["tapioca_starch"])

(:Synonym
  name      : tapioca starch
  source    : RxNav_DRUGBANK
  id        : syn_c400feed53af0474a0b36fde1982a97ffefafacbe46e330191ae4496892f8018})
      │
      └── [:HAS_SYNONYM]
                  │
                  ▼
(:Ingredient
  name      : tapioca starch allergenic extract
  source    : RxNorm
  rxcui     : 901346})

(:Synonym
  name      : tapioca starch
  source    : RxNav_RXNORM
  id        : syn_06436deda93fcb52aef6b12e878d1aa435420b0cb32209d0b81f2b3a2e3e7b91})
      │
      └── [:HAS_SYNONYM]
                  │
                  ▼
(:Ingredient
  name      : starch
  source    : RxNorm
  rxcui     : 2619252})



![tapioca_starch](../../datasets/knowledge_graph/images/tapioca_starch.png)

## DDI Analysis

In [14]:
import pandas as pd

ddi_df = pd.read_csv(DDI_PATH)
print(f"Total DDI Rows: {ddi_df.shape[0]}")

skipped_ddi_df = pd.read_csv(SKIPPED_DDI_RECORD_LOG_PATH)
print(f"Total DDI Skipped Rows: {skipped_ddi_df.shape[0]}")

print(f"Total DDI Resolved Rows: {ddi_df.shape[0] - skipped_ddi_df.shape[0]}")

ddi_resolved = (ddi_df.shape[0] - skipped_ddi_df.shape[0])/ddi_df.shape[0]
print(f"Total DDI Resolved Rows (in Percentage): {(ddi_resolved * 100):.2f}%")


Total DDI Rows: 191541
Total DDI Skipped Rows: 0
Total DDI Resolved Rows: 191541
Total DDI Resolved Rows (in Percentage): 100.00%


## Brand Analysis

In [15]:
import pandas as pd

brand_df = pd.read_csv(BPOM_BRAND_PATH)
print(f"Total Brand Rows: {brand_df.shape[0]}")

unresolved_ingredients_brand_df = pd.read_csv(UNRESOLVED_BRAND_ING_LOG_PATH)
df_grouped = (
    unresolved_ingredients_brand_df.groupby("brand_name", as_index=False)
      .agg({
          "unresolved_ingredient": list,
          "raw_composition": "first",
      })
)
print(f"Total Unresolved Brand Rows: {df_grouped.shape[0]}")
print(f"Total DDI Resolved Rows: {brand_df.shape[0] - df_grouped.shape[0]}")

brand_resolved = (brand_df.shape[0] - df_grouped.shape[0])/brand_df.shape[0]
print(f"Total Brand Ingredients Resolved Rows (in Percentage): {(brand_resolved * 100):.2f}%")


Total Brand Rows: 9257
Total Unresolved Brand Rows: 2227
Total DDI Resolved Rows: 7030
Total Brand Ingredients Resolved Rows (in Percentage): 75.94%


## Knowledge Graph Analysis

Querying the longest relation KG
```text
MATCH (syn1:Synonym)<-[:HAS_SYNONYM]-(pin1:PreciseIngredient)-[:FORM_OF]->(ing1:Ingredient)
      <-[:HAS_PARTICIPANT]-(ddi:DDI)-[:HAS_PARTICIPANT]->
      (ing2:Ingredient)<-[:FORM_OF]-(pin2:PreciseIngredient)-[:HAS_SYNONYM]->(syn2:Synonym)
WHERE syn1 <> syn2 AND ing1 <> ing2
RETURN syn1, pin1, ing1, ddi, ing2, pin2, syn2
LIMIT 1;
```

Resulting graph below:
![longest_graph](../../datasets/reports/longest_graph.png)